# 04 — Export, and the ship gate

## The rule this notebook enforces

> Python numbers are for **iteration**. Any number that would change a decision
> is confirmed through `npx tsx scripts/eval.ts` before it is written down.

The parity gate shows the two agree on everything checkable — 287/287 identical
ranks on a shared cell. This notebook is where a candidate stops being an
experiment and gets measured on the served path, in the serving runtime, with
the serving artifact.

That distinction is not ceremony. The Python loop scans a local cell exactly;
production queries an IVFFlat index approximately, through ONNX, over 693,325
rows rather than WordNet's 117,791. Those differences are small and *measured*
(the approximate index costs ~0.3pp) — but "small and measured" is a thing you
establish, not assume.

In [ ]:
import sys, os
from pathlib import Path

# rdlib lives at training/rdlib; this notebook is at training/notebooks.
sys.path.insert(0, str(Path.cwd().parent))

# Cells are large and the darwin default lives under os.tmpdir(), which gets
# reaped. Point this somewhere durable and OUTSIDE the repo -- the working tree
# is in OneDrive, which would try to sync ~170 MB per cell.
os.environ.setdefault("EVAL_CELL_DIR", str(Path.home() / "rd_eval_cells"))

import rdlib
from rdlib import paths
print("repo      ", paths.REPO_ROOT)
print("cells     ", paths.cell_dir())

## Step 1 — export to ONNX

`lib/embedder.ts` loads `onnx/model.onnx` at full precision (`quantized: false`)
and applies `{ pooling: "mean", normalize: true }`. The export has to reproduce
that pipeline exactly, or query vectors stop matching stored ones.

**There is no quantized artifact, and that is not a tuning choice** —
`onnx/model_quantized.onnx` returns a 15-byte "Entry not found" from HF.
Producing one would change query vectors and needs a full `eval:prod` behind it.

In [ ]:
from pathlib import Path
from rdlib.paths import ARTIFACTS_DIR

# Point this at the checkpoint from notebook 02.
CANDIDATE = ARTIFACTS_DIR / "rd22_biencoder_wikt_paraphrase" / "final"
EXPORT_DIR = ARTIFACTS_DIR / "onnx_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

assert CANDIDATE.exists(), f"no checkpoint at {CANDIDATE} -- run notebook 02 first"
print("exporting", CANDIDATE)

In [ ]:
import subprocess, sys

# optimum-cli exports the TRANSFORMER only. The Pooling and Normalize layers are
# applied by the consumer -- Transformers.js does mean-pool + normalise itself
# via { pooling: "mean", normalize: true }, which is exactly why the served
# pipeline reproduces sentence-transformers without those layers in the graph.
cmd = [
    sys.executable, "-m", "optimum.commands.optimum_cli",
    "export", "onnx",
    "--model", str(CANDIDATE),
    "--task", "feature-extraction",
    str(EXPORT_DIR),
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout[-2000:] or proc.stderr[-2000:])
print("\nfiles:", sorted(p.name for p in EXPORT_DIR.iterdir()))

## Step 2 — verify the export before trusting it

An ONNX export that is subtly wrong produces plausible vectors and a quietly
worse model. Compare the exported graph against the PyTorch model it came from,
applying mean pooling and L2 normalisation by hand so the comparison covers the
whole served pipeline rather than just the encoder block.

The reference for "close enough" is the measurement `parity.check_encoder()`
already makes on the shipped model: **cos = 1.0000001, max abs diff 1.3e-07**.

In [ ]:
import numpy as np, onnxruntime as ort
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

TEXTS = [
    "the smell of rain on dry earth",
    "when you say a word so many times it stops sounding like a word",
    "a wrong action attributable to bad judgment or ignorance or inattention",
]

tok = AutoTokenizer.from_pretrained(str(CANDIDATE))
sess = ort.InferenceSession(str(EXPORT_DIR / "model.onnx"), providers=["CPUExecutionProvider"])
st = SentenceTransformer(str(CANDIDATE), device="cpu")

enc = tok(TEXTS, padding=True, truncation=True, return_tensors="np")
feeds = {i.name: enc[i.name] for i in sess.get_inputs() if i.name in enc}
hidden = sess.run(None, feeds)[0]

# Transformers.js's mean_pooling() is a plain attention-masked mean, then L2.
mask = enc["attention_mask"][..., None].astype(np.float32)
pooled = (hidden * mask).sum(1) / np.clip(mask.sum(1), 1e-9, None)
onnx_vecs = pooled / np.linalg.norm(pooled, axis=1, keepdims=True)

torch_vecs = st.encode(TEXTS, normalize_embeddings=True)

print(f"{'cos':>14} {'max abs diff':>14}   text")
ok = True
for t, a, b in zip(TEXTS, onnx_vecs, torch_vecs):
    cos = float(np.asarray(a, np.float64) @ np.asarray(b, np.float64))
    md = float(np.abs(np.asarray(a, np.float64) - np.asarray(b, np.float64)).max())
    ok &= md < 1e-5
    print(f"{cos:>14.9f} {md:>14.2e}   {t[:40]}")
print("\nPASS" if ok else "\nFAIL -- do not ship this export")

## Step 3 — the layout `lib/embedder.ts` expects

`env.localModelPath` defaults relative to the Transformers.js module directory,
**not** `process.cwd()`, so the on-disk layout must mirror the repo id:

```
models/<org>/<name>/config.json
models/<org>/<name>/tokenizer.json
models/<org>/<name>/tokenizer_config.json
models/<org>/<name>/onnx/model.onnx
```

`scripts/fetch-model.mjs` writes exactly those four files, and
`next.config.js`'s `outputFileTracingIncludes` traces `models/**` into the
`/api/lookup` function. If you swap the model, `fetch-model.mjs`'s `FILES` list
(names **and byte sizes**) has to change with it.

The cell below stages a candidate locally so `eval.ts` can score it. It writes
to `training/artifacts/`, **not** to `models/` — overwriting the served model in
place would make every later comparison ambiguous about what it measured.

In [ ]:
import shutil

ORG, NAME = "rd22", "candidate"
staged = ARTIFACTS_DIR / "models" / ORG / NAME
(staged / "onnx").mkdir(parents=True, exist_ok=True)

for src, dst in [
    (EXPORT_DIR / "model.onnx", staged / "onnx" / "model.onnx"),
    (CANDIDATE / "config.json", staged / "config.json"),
    (CANDIDATE / "tokenizer.json", staged / "tokenizer.json"),
    (CANDIDATE / "tokenizer_config.json", staged / "tokenizer_config.json"),
]:
    if src.exists():
        shutil.copy2(src, dst)
        print(f"  {dst.relative_to(ARTIFACTS_DIR)}  ({dst.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"  MISSING {src}")

## Step 4 — the authoritative number

Build a cell with the candidate, then score it with the **TypeScript harness**.
`eval.ts` reads the encoder from the cell's `meta.model`, so the model has to be
loadable by Transformers.js. Two routes:

- **push to the Hub** and pass the repo id — closest to how production loads it;
- **point at the staged directory** — needs `allowLocalModels` in
  `scripts/lib/embedModel.ts`, which is currently `false` because `env` is a
  process-wide singleton shared with `lib/embedder.ts`. Change it at the call
  site, never at module scope: both modules setting it at import time is exactly
  what once broke `eval:prod` with *"both local and remote models are disabled"*.

```bash
EVAL_CELL_DIR=~/rd_eval_cells npx tsx scripts/eval.ts \
    --set eval/sets/v1.jsonl \
    --index-file cell_rd22_biencoder_wikt_paraphrase \
    --tag rd22_candidate

npx tsx scripts/eval.ts --compare \
    eval/runs/full_gloss_ft.json eval/runs/rd22_candidate.json
```

In [ ]:
from rdlib import runs as R
from rdlib.metrics import score, compare

def head(run):
    return [r for r in run.results if r.source == "authored" and r.meta.get("reachable")]

try:
    control = R.load_run("full_gloss_ft")      # RD-16's control, 25.4% lenient
    cand = R.load_run("rd22_candidate")
except FileNotFoundError as exc:
    print(f"{exc}\n\nRun the eval.ts commands above first.")
else:
    mc, mk = score(head(control)), score(head(cand))
    c = compare(head(control), head(cand), lenient=True)
    print(f"{'':16}{'control':>10}{'candidate':>11}")
    print(f"  lenient R@1 {mc.lenient_recall1*100:>9.1f}%{mk.lenient_recall1*100:>10.1f}%")
    print(f"  strict  R@1 {mc.recall1*100:>9.1f}%{mk.recall1*100:>10.1f}%")
    print(f"  R@10        {mc.recall10*100:>9.1f}%{mk.recall10*100:>10.1f}%")
    print(f"  echo        {mc.echo_rate*100:>9.1f}%{mk.echo_rate*100:>10.1f}%")
    print(f"\n  delta {c['delta_pp']:+.1f}pp   {c['n_wins']}W/{c['n_regressions']}R   p={c['p']:.4f}")
    print(f"  9a: {'CLEARS THE BAR' if c['clears_9a_bar'] else 'NULL RESULT -- do not act on it'}")

## Step 5 — the ship gate

RD-16's candidate failed on **three independent conditions**, and any one of them
is disqualifying. Check all three, not just the first.

| # | condition | why |
|---|---|---|
| 1 | **≥ ~6pp lenient R@1**, paired, on the authored-reachable slice | METHODS §9a. Deliberately not renegotiated when its original anchor was superseded |
| 2 | **384 dimensions** | `GlossEmbedding` is `halfvec(384)`. 768-dim cannot fit |
| 3 | **Fits the function bundle** | `/api/lookup` traces at 151.6 MB against a 250 MB limit; the current ONNX is 86 MB |
| 4 | **Echo has not risen** | a gain bought by lexical overlap is the thing this project removed |

In [ ]:
import onnx

model_path = EXPORT_DIR / "model.onnx"
size_mb = model_path.stat().st_size / 1e6
dim = onnx.load(str(model_path)).graph.output[0].type.tensor_type.shape.dim[-1].dim_value

print(f"  dim              {dim}      {'OK' if dim == 384 else 'BLOCKS SHIPPING -- halfvec(384)'}")
print(f"  onnx size        {size_mb:.1f} MB   "
      f"{'OK' if size_mb < 120 else 'CHECK the 250 MB bundle limit'}")
print(f"  current model    86.0 MB (for comparison)")

## If it ships, what actually has to happen

Scoring well is the beginning, not the end. The full path:

1. **Re-embed all 693,325 `GlossEmbedding` rows** with the new encoder. Query
   and document vectors must come from the same model — a partial migration
   leaves two vector spaces in one table and is silently wrong.
2. **Rebuild the IVFFlat index**, with `SET maintenance_work_mem` **in the
   migration file**. Neon's default is 64 MB; the build at `lists = 833` needs
   169 MB and fails outright without it. `prisma db execute --file` runs the
   file as one transaction, so a failed `CREATE INDEX` rolls its `DROP INDEX`
   back with it and the serving index survives — safe to retry.
3. **Re-sweep `probes` and `lists` together.** They are one setting with two
   halves: a probe scans roughly `rows / lists`, so a changed row count changes
   what a given `probes` value means.
4. **Update `scripts/fetch-model.mjs`** — the `FILES` list carries exact byte
   sizes and a mismatch fails the build by design.
5. **Re-run `npm run verify-viz`.** `/explain` projects retrieved synsets onto a
   PCA basis fitted to the *old* vectors; a new encoder invalidates
   `public/viz/pipeline-snapshot.json`, which needs `npm run build-viz`.
6. **File the ticket**, positive or negative. RD-12 and RD-16 are both negative
   and both saved later work — that is what the backlog is for.